# 02. Huấn luyện Mô hình & Thử nghiệm (Model Training & Experimentation)

**Thành viên phụ trách:** Thành viên 2 - Model Development & Training

**Mục tiêu:**
1. Nạp dữ liệu NBA đã xử lý (`train.csv`, `val.csv`, `test.csv`) từ `data/processed/`.
2. Khởi tạo 4 mô hình học máy theo thiết kế:
   - `DummyClassifier(strategy='prior')`: Mô hình baseline cơ sở.
   - `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`: Tuyến tính, dễ diễn giải.
   - `RandomForestClassifier(class_weight='balanced', random_state=42)`: Cây quyết định kết hợp ensemble bagging.
   - `HistGradientBoostingClassifier(random_state=42)`: Thuật toán tăng cường độ dốc hiệu năng cao.
3. Huấn luyện các mô hình trên tập Train (mùa giải $\le 2016$).
4. Dự báo xác suất (`y_proba`) và xuất file `reports/predictions.csv` có đúng 6 cột:
   `['player', 'season', 'split', 'model', 'y_true', 'y_proba']`.
5. Lưu trữ các mô hình đã fit vào thư mục `models/` bằng `joblib`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
from src.train import (
    get_default_models,
    train_models,
    export_predictions,
    save_models
)

print("Imports thành công từ src.train!")

## Bước 1: Nạp dữ liệu từ `data/processed/`
Chọn lọc danh sách 24 đặc trưng số liệu bao gồm các chỉ số Z-Score theo mùa, chỉ số cá nhân, chỉ số nâng cao, tỷ lệ thắng đội và số lần All-Star quá khứ.

In [ ]:
processed_dir = ROOT_DIR / "data" / "processed"

train_df = pd.read_csv(processed_dir / "train.csv")
val_df = pd.read_csv(processed_dir / "val.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

target_col = "All_Star"

feature_cols = [
    'pts_per_game_zscore', 'ast_per_game_zscore', 'trb_per_game_zscore',
    'per_zscore', 'ws_zscore', 'vorp_zscore',
    'team_win_pct', 'prev_allstar_count',
    'mp_per_game', 'pts_per_game', 'ast_per_game', 'trb_per_game',
    'stl_per_game', 'blk_per_game', 'tov_per_game',
    'fg_percent', 'x3p_percent', 'ft_percent',
    'per', 'ts_percent', 'usg_percent', 'ws', 'bpm', 'vorp'
]
feature_cols = [c for c in feature_cols if c in train_df.columns]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Số lượng đặc trưng sử dụng ({len(feature_cols)}): {feature_cols}")
print(f"Kích thước các tập: Train={train_df.shape}, Val={val_df.shape}, Test={test_df.shape}")
print(f"Tỷ lệ All-Star trên tập Train: {y_train.mean():.2%}")

## Bước 2: Khởi tạo 4 mô hình phân loại (`get_default_models`)

In [ ]:
models = get_default_models(random_state=42)
for name, m in models.items():
    print(f"- {name}: {m}")

## Bước 3: Huấn luyện các mô hình trên tập Train (`train_models`)

In [ ]:
print("Bắt đầu huấn luyện 4 mô hình trên 11.473 dòng dữ liệu...")
fitted_models = train_models(models, X_train, y_train)
print("Hoàn tất huấn luyện thành công!")

## Bước 4: Xuất kết quả dự đoán ra `reports/predictions.csv` (`export_predictions`)
Định dạng xuất chuẩn gồm đúng 6 cột: `['player', 'season', 'split', 'model', 'y_true', 'y_proba']`.

In [ ]:
reports_dir = ROOT_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
predictions_path = reports_dir / "predictions.csv"

# Dự đoán tập Validation (2017 - 2019)
val_preds = export_predictions(
    models_dict=fitted_models,
    X_df=val_df,
    y_true=y_val,
    split_name="val",
    feature_cols=feature_cols
)

# Dự đoán tập Test (2020 - 2026)
test_preds = export_predictions(
    models_dict=fitted_models,
    X_df=test_df,
    y_true=y_test,
    split_name="test",
    feature_cols=feature_cols
)

# Hợp nhất và lưu ra CSV
all_preds = pd.concat([val_preds, test_preds], ignore_index=True)
all_preds.to_csv(predictions_path, index=False)

print(f"Đã lưu kết quả dự đoán ({len(all_preds)} dòng) vào: {predictions_path}")
print("6 cột chuẩn:", list(all_preds.columns))
all_preds.head(8)

## Bước 5: Lưu trữ mô hình vào thư mục `models/` (`save_models`)

In [ ]:
models_dir = ROOT_DIR / "models"
saved_paths = save_models(fitted_models, models_dir)
for name, p in saved_paths.items():
    print(f"Đã lưu mô hình: {name} -> {p.name}")